<a href="https://colab.research.google.com/github/rachmadnanda/MachineLearning2026/blob/main/jobsheet_pbl/JS09_PBL_244107020095_RACHMAD_FEBRIANANDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS09 — Klasifikasi Kategori Harga Properti: k-NN dan Naive Bayes

Jobsheet asli memakai Iris, data dummy, dan SMS spam. Versi PBL ini mengklasifikasikan
properti ke **Murah / Sedang / Mahal** berdasarkan atribut fisik dan frekuensi lokasi.

Karena label dibentuk dari kuantil harga, ketiga kelas berjumlah hampir sama
(±18,7 ribu per kelas), sehingga *baseline* akurasi acak adalah **1/3 ≈ 0,333**.

## Langkah 1 — Muat Data, Bentuk Label, dan Bagi Data

Split 70:30 dengan `stratify` memastikan proporsi kelas di data latih dan uji sama.
Standardisasi dilakukan **setelah** split (di dalam pipeline) agar statistik data uji
tidak bocor ke proses latih.

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42

df = pd.read_csv('dataset/properti_multi_kota.csv').drop_duplicates()
df = df[(df['luas_m2'] >= 10) & (df['luas_m2'] <= 2000)]
df = df[df['luas_bangunan'].isna() |
        ((df['luas_bangunan'] >= 10) & (df['luas_bangunan'] <= 2000))]
df = df[df['harga'].between(50_000_000, 100_000_000_000)]
df = df[df['jumlah_kamar'].isna() | (df['jumlah_kamar'] <= 20)]
df = df[df['jumlah_kamar_mandi'].isna() | (df['jumlah_kamar_mandi'] <= 20)]

FITUR_NUM = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']
df[FITUR_NUM] = df[FITUR_NUM].fillna(df[FITUR_NUM].median())
df['lokasi_bersih'] = df['lokasi'].str.strip().str.title()
df['lokasi_freq'] = df['lokasi_bersih'].map(df['lokasi_bersih'].value_counts(normalize=True))

q1, q2 = df['harga'].quantile([1 / 3, 2 / 3])
df['kelas_harga'] = np.where(df['harga'] < q1, 'Murah',
                     np.where(df['harga'] <= q2, 'Sedang', 'Mahal'))

FITUR = FITUR_NUM + ['lokasi_freq']
X = df[FITUR]
y = df['kelas_harga']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(y_train.value_counts().to_string())

Train: 39,319 | Test: 16,851
kelas_harga
Sedang    13109
Murah     13106
Mahal     13104


## Langkah 2 — k-NN dengan `k = 3`

k-NN memprediksi kelas berdasarkan mayoritas $k$ tetangga terdekat. Karena memakai
jarak, standardisasi wajib: tanpa standardisasi, `luas_m2` (puluhan–ribuan) akan
mendominasi `jumlah_kamar` (1–10).

In [2]:
knn3 = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3, n_jobs=-1))

t0 = time.perf_counter()
knn3.fit(X_train, y_train)
pred3 = knn3.predict(X_test)
waktu3 = time.perf_counter() - t0

print(f'Akurasi k=3 : {accuracy_score(y_test, pred3):.4f}')
print(f'Waktu       : {waktu3:.1f} detik')
print()
print(classification_report(y_test, pred3, digits=4))

Akurasi k=3 : 0.7547
Waktu       : 0.2 detik

              precision    recall  f1-score   support

       Mahal     0.7953    0.8296    0.8121      5616
       Murah     0.8011    0.7953    0.7982      5617
      Sedang     0.6629    0.6392    0.6508      5618

    accuracy                         0.7547     16851
   macro avg     0.7531    0.7547    0.7537     16851
weighted avg     0.7531    0.7547    0.7537     16851



In [3]:
cm = confusion_matrix(y_test, pred3, labels=['Murah', 'Sedang', 'Mahal'])
ConfusionMatrixDisplay(cm, display_labels=['Murah', 'Sedang', 'Mahal']).plot(cmap='Blues')
plt.title('Confusion matrix k-NN (k=3)')
plt.tight_layout()
plt.show()

## Langkah 3 — Mencari Nilai `k` Terbaik

Nilai `k` terlalu kecil membuat model sensitif terhadap noise; terlalu besar membuat
batas kelas terlalu halus (bias). Pencarian memakai 5-fold CV pada **sampel 15.000**
baris agar cepat, dengan `k = 1..30`.

In [4]:
sampel = X_train.sample(15_000, random_state=RANDOM_STATE)
y_sampel = y_train.loc[sampel.index]

ks = list(range(1, 31, 2)) + [40, 50]
skor = []
for k in ks:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k, n_jobs=-1))
    cv = cross_val_score(model, sampel, y_sampel,
                         cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                         scoring='accuracy').mean()
    skor.append(cv)

tabel_k = pd.DataFrame({'k': ks, 'akurasi CV': skor}).sort_values('akurasi CV', ascending=False)
print(tabel_k.head(8).to_string(index=False))

plt.figure(figsize=(8, 4))
plt.plot(ks, skor, 'o-')
plt.xlabel('k')
plt.ylabel('Akurasi CV')
plt.title('Pemilihan k untuk k-NN')
plt.tight_layout()
plt.show()

 k  akurasi CV
 9    0.736067
 7    0.735467
15    0.734600
17    0.734000
 5    0.733667
19    0.733200
21    0.732867
13    0.732667


## Langkah 4 — k-NN dengan `k` Terbaik

Model dilatih ulang pada seluruh data latih dan dievaluasi pada data uji yang sama.

In [5]:
k_terbaik = int(tabel_k.iloc[0]['k'])
knn_best = make_pipeline(StandardScaler(),
                         KNeighborsClassifier(n_neighbors=k_terbaik, n_jobs=-1))
t0 = time.perf_counter()
knn_best.fit(X_train, y_train)
pred_best = knn_best.predict(X_test)
waktu_best = time.perf_counter() - t0

print(f'k terbaik   : {k_terbaik}')
print(f'Akurasi     : {accuracy_score(y_test, pred_best):.4f}')
print(f'Waktu       : {waktu_best:.1f} detik')

k terbaik   : 9
Akurasi     : 0.7554
Waktu       : 0.3 detik


## Langkah 5 — Naive Bayes

Naive Bayes mengasumsikan fitur saling independen. Untuk fitur kontinu dipakai
**GaussianNB**; untuk fitur yang dibinerkan (di atas median) dipakai **BernoulliNB**.
**MultinomialNB** tidak dipakai karena dirancang untuk data hitungan (seperti frekuensi
kata), bukan atribut fisik properti.

In [6]:
hasil_nb = []

gnb = make_pipeline(StandardScaler(), GaussianNB())
t0 = time.perf_counter()
gnb.fit(X_train, y_train)
pred_gnb = gnb.predict(X_test)
hasil_nb.append({'Model': 'GaussianNB', 'Akurasi': accuracy_score(y_test, pred_gnb),
                 'Waktu (s)': time.perf_counter() - t0})

# BernoulliNB: semua fitur dibinerkan dengan ambang median data latih
median = X_train.median()
X_train_bin = (X_train > median).astype(int)
X_test_bin = (X_test > median).astype(int)

bnb = BernoulliNB()
t0 = time.perf_counter()
bnb.fit(X_train_bin, y_train)
pred_bnb = bnb.predict(X_test_bin)
hasil_nb.append({'Model': 'BernoulliNB', 'Akurasi': accuracy_score(y_test, pred_bnb),
                 'Waktu (s)': time.perf_counter() - t0})

# MultinomialNB hanya untuk perbandingan: fitur digeser agar tidak negatif
mnb = MultinomialNB()
mnb.fit(X_train - X_train.min() + 1e-6, y_train)
pred_mnb = mnb.predict(X_test - X_train.min() + 1e-6)
hasil_nb.append({'Model': 'MultinomialNB (tidak cocok)', 'Akurasi': accuracy_score(y_test, pred_mnb),
                 'Waktu (s)': np.nan})

pd.DataFrame(hasil_nb).round(4)

,Model,Akurasi,Waktu (s)
0,GaussianNB,0.5923,0.0721
1,BernoulliNB,0.6549,0.0940
2,MultinomialNB (tidak cocok),0.5508,NaN


## Langkah 6 — Perbandingan Akhir

Semua model dibandingkan pada data uji yang sama, ditambah `DummyClassifier`
(menebak kelas mayoritas) sebagai batas bawah.

In [7]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)

ringkasan = pd.DataFrame([
    {'Model': 'Dummy (mayoritas)', 'Akurasi': accuracy_score(y_test, pred_dummy)},
    {'Model': 'k-NN k=3', 'Akurasi': accuracy_score(y_test, pred3)},
    {'Model': f'k-NN k={k_terbaik}', 'Akurasi': accuracy_score(y_test, pred_best)},
    {'Model': 'GaussianNB', 'Akurasi': accuracy_score(y_test, pred_gnb)},
    {'Model': 'BernoulliNB', 'Akurasi': accuracy_score(y_test, pred_bnb)},
]).sort_values('Akurasi', ascending=False)

print(ringkasan.round(4).to_string(index=False))

plt.figure(figsize=(8, 3.8))
sns.barplot(data=ringkasan, x='Akurasi', y='Model', palette='viridis')
plt.axvline(1 / 3, color='red', linestyle='--', label='acak (1/3)')
plt.xlim(0, 1)
plt.legend()
plt.title('Akurasi model klasifikasi kategori harga')
plt.tight_layout()
plt.show()

            Model  Akurasi
         k-NN k=9   0.7554
         k-NN k=3   0.7547
      BernoulliNB   0.6549
       GaussianNB   0.5923
Dummy (mayoritas)   0.3334


## Langkah 7 — Tugas: Pengaruh Seleksi Fitur pada k-NN

Jobsheet asli meminta pencarian fitur paling optimal pada `voice.csv`. Di sini tugasnya:
bandingkan k-NN dengan empat kombinasi fitur.

In [8]:
kombinasi = {
    'luas_m2 saja': ['luas_m2'],
    'luas tanah + bangunan': ['luas_m2', 'luas_bangunan'],
    'semua numerik': FITUR_NUM,
    'numerik + lokasi_freq': FITUR,
}

baris = []
for nama, kolom in kombinasi.items():
    model = make_pipeline(StandardScaler(),
                          KNeighborsClassifier(n_neighbors=k_terbaik, n_jobs=-1))
    model.fit(X_train[kolom], y_train)
    baris.append({'Fitur': nama, 'Jumlah': len(kolom),
                  'Akurasi': accuracy_score(y_test, model.predict(X_test[kolom]))})

pd.DataFrame(baris).sort_values('Akurasi', ascending=False).round(4)

,Fitur,Jumlah,Akurasi
3,numerik + lokasi_freq,5,0.7554
2,semua numerik,4,0.6835
1,luas tanah + bangunan,2,0.6677
0,luas_m2 saja,1,0.5025


**Analisis JS09:**

- **k-NN `k = 3`** memberi akurasi **0,7547** pada data uji — lebih dari dua kali *baseline* acak 0,333 dan **lebih tinggi daripada regresi logistik JS03 (0,68)**. Pencarian `k` menemukan **k = 9** dengan akurasi **0,7554**, hanya sedikit lebih baik; kurva akurasi terhadap `k` mendatar setelah k ≈ 7.
- Kelas **Sedang** paling sulit: precision 0,663 / recall 0,639, jauh di bawah Mahal (0,795/0,830) dan Murah (0,801/0,795). Properti menengah secara fisik mirip dengan kedua ekstrem sehingga paling banyak salah klasifikasi.
- **GaussianNB** hanya 0,5923 dan **BernoulliNB** 0,6549. Asumsi independensi fitur dilanggar (`luas_m2` dan `luas_bangunan` berkorelasi), dan binning pada BernoulliNB menghilangkan besaran luas. **MultinomialNB** 0,5508 — memang tidak cocok untuk data kontinu.
- `lokasi_freq` sangat penting: k-NN dengan numerik + frekuensi lokasi mencapai **0,7554**, sedangkan numerik saja **0,6835**, luas tanah + bangunan 0,6677, dan `luas_m2` saja 0,5025. Lokasi menyumbang sekitar **+7 poin akurasi**.
- Dibandingkan model lain: k-NN (0,755) sedikit di atas SVM RBF (0,720) dan MLP (JS13), tetapi di bawah potensi model pohon. k-NN tetap mahal saat prediksi karena harus menghitung jarak ke seluruh data latih.
- Kesimpulan: k-NN adalah baseline kuat dan murah untuk data ini, tetapi model ensemble/pohon (JS15) lebih praktis untuk produksi karena waktu inference-nya konstan dan mampu menangani kategori lokasi penuh.
